In [3]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import math
import re

import sys
sys.path.append('../../..')
# from mount_drive import mount_s_drive

In [4]:
# mount_s_drive(subfolder='LCICM/Databases/ACCMPMAP')

In [5]:
data_dir = '/projects/LCICM/Databases/ACCMPMAP/'

In [6]:
def read_by_chunks(file_path,chunk_size=1e6,head='infer'):
    df_chunks = []
    num_chunks = 0
    
    for chunk in pd.read_csv(file_path,chunksize=chunk_size,header=head):
        #chunk = chunk[chunk[0].isin(patient_ids)]
        df_chunks.append(chunk)
        if num_chunks % 20 == 0:
            print(f'Chunk {num_chunks} Processed.')
        num_chunks += 1
        del chunk
    print('Processing finished.')
    
    return pd.concat(df_chunks, ignore_index=True)

In [7]:
def nameSearchCardiacArrest(text):
    text = str(text).lower()
    patterns = [r'cardia.*rrest',
                r'cardio.*rrest',
                r'circulat.*rrest',
                r'\basystole',
                r'\basystolia',
                r'\bpea\b|pulseless elec.*act.*']
    if any(re.search(pattern, text) for pattern in patterns):
        exclusion_patterns = [r'history|hx|h/o',
                              r'before cardiac arrest',
                              r'due to cardiac arrest',
                              r'neonatal',
                              r'newborn']       
        if not any(re.search(pattern, text) for pattern in exclusion_patterns):
            return True   
    return False

def icdSearchCardiacArrest(text):
    text = str(text).lower()
    icd10_match = re.search(r'\bi46\.\b', text)
    icd9_match = re.search(r'\b427\.5\b', text)
    return bool(icd10_match or icd9_match)

In [8]:
patients_df = pd.read_csv(data_dir+'2024-08-15/dbo.accm_patient.csv')
inpatient_df = read_by_chunks(data_dir+'2024-08-15/dbo.accm_inpatient.csv')
icustay_df = pd.read_csv(data_dir+'core_icustay_8_15_2024.csv')
encounter_dx_df = read_by_chunks(data_dir+'2024-08-15/dbo.accm_encounter_dx.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/projects/LCICM/Databases/ACCMPMAP/2024-08-15/dbo.accm_patient.csv'

In [13]:
patients_df1 = patients_df[['osler_id','gender','birth_date']]
inpatient_df1 = inpatient_df[['osler_id','pat_enc_csn_id','adt_pat_class_c','adt_arrival_time','hosp_admsn_time','hosp_disch_time','admit_source_c','hosp_admsn_type_c',
                              'ed_visit_yn','emer_adm_date','dep_speciality','hospital_service','disch_disp_c']]
icustay_df1 = icustay_df[['osler_id','pat_enc_csn_id','in_time','out_time','seq','icu','origin']]
encounter_dx_df1 = encounter_dx_df[['osler_id','pat_enc_csn_id','dx_name','icd10_code','icd9_code','enc_contact_date','primary_dx_yn','dx_ed_yn','dx_qualifier','annotation']]

### Cardiac Arrest

In [14]:
encounter_dx_df1['name_search'] = encounter_dx_df1['dx_name'].transform(nameSearchCardiacArrest)
encounter_dx_df1['icd9_icd10'] = encounter_dx_df1['icd9_code'] + ' ' + encounter_dx_df1['icd10_code']
encounter_dx_df1['icd_search'] = encounter_dx_df1['icd9_icd10'].transform(icdSearchCardiacArrest)
ca_df = encounter_dx_df1[(encounter_dx_df1['name_search'])|(encounter_dx_df1['icd_search'])]

/tmp/ipykernel_188/2283379023.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  encounter_dx_df1['name_search'] = encounter_dx_df1['dx_name'].transform(nameSearchCardiacArrest)
/tmp/ipykernel_188/2283379023.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  encounter_dx_df1['icd9_icd10'] = encounter_dx_df1['icd9_code'] + ' ' + encounter_dx_df1['icd10_code']
/tmp/ipykernel_188/2283379023.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_in

In [15]:
ca_df_tmp = ca_df[['osler_id','pat_enc_csn_id','enc_contact_date','dx_name','primary_dx_yn','dx_ed_yn','dx_qualifier','annotation']]
ca_df_tmp['primary_dx_yn'] = ca_df_tmp.groupby(['pat_enc_csn_id'])['primary_dx_yn'].transform(lambda x: 'Y' if 'Y' in x.values else 'N')
ca_df_tmp['dx_ed_yn'] = ca_df_tmp.groupby(['pat_enc_csn_id'])['dx_ed_yn'].transform(lambda x: 'Y' if 'Y' in x.values else 'N')

/tmp/ipykernel_188/632480073.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ca_df_tmp['primary_dx_yn'] = ca_df_tmp.groupby(['pat_enc_csn_id'])['primary_dx_yn'].transform(lambda x: 'Y' if 'Y' in x.values else 'N')
/tmp/ipykernel_188/632480073.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ca_df_tmp['dx_ed_yn'] = ca_df_tmp.groupby(['pat_enc_csn_id'])['dx_ed_yn'].transform(lambda x: 'Y' if 'Y' in x.values else 'N')


In [16]:
ca_df_tmp['pat_enc_csn_id'].nunique()

8899

### OHCA

In [17]:
ip_df = pd.merge(patients_df1, inpatient_df1, on='osler_id', how='inner')
ca_ip_df = pd.merge(ca_df_tmp, ip_df, on=['osler_id','pat_enc_csn_id'], how='inner')
ca_ip_df['birth_date'] = pd.to_datetime(ca_ip_df['birth_date'], errors='coerce')
ca_ip_df['hosp_admsn_time'] = pd.to_datetime(ca_ip_df['hosp_admsn_time'], errors='coerce')
ca_ip_df = ca_ip_df.dropna(subset=['hosp_admsn_time'])
ca_ip_df['age'] = (ca_ip_df['hosp_admsn_time']-ca_ip_df['birth_date']).apply(lambda x: math.floor(x.days/365.25))
ca_ip_df = ca_ip_df[ca_ip_df['age'] >= 18]
ca_ip_df.loc[ca_ip_df['age'] > 100, 'age'] = 89  # match eICU
ca_ip_icu_df = pd.merge(ca_ip_df, icustay_df1, on=['osler_id','pat_enc_csn_id'], how='left')
ca_ip_icu_df = ca_ip_icu_df[ca_ip_icu_df['annotation']!='H/o cardiac arrest']

In [21]:
df = ca_ip_icu_df[(ca_ip_icu_df['seq']==1.0)|(ca_ip_icu_df['seq'].isna())]
df1 = df[df['hosp_admsn_type_c']!=3.0] # admission reason is non elective
df2 = df1[(df1['ed_visit_yn']=='Y')|(df1['origin']=='init')] # went through ED or was admitted straight to ICU
df3 = df2[(df2['dx_ed_yn']=='Y')|(df2['origin']=='init')] # diagnosis observed in ED or admitted straight to ICU

# merge dx_name
dx_name_df = df3.groupby('pat_enc_csn_id').agg({'dx_name': lambda x: '; '.join(x.unique())})
df3 = df3.drop(columns=['dx_name'])
df3 = pd.merge(df3, dx_name_df, on='pat_enc_csn_id', how='left')
df3 = df3.drop_duplicates(subset=['osler_id','pat_enc_csn_id'])

# drop patients with multiple encounters
grouped = df3.groupby('osler_id')['pat_enc_csn_id'].nunique()
duplicate_osler_ids = grouped[grouped > 1].index
df4 = df3[~df3['osler_id'].isin(duplicate_osler_ids)]

In [23]:
print('Cardiac arrest diagnoses:', ca_df_tmp['pat_enc_csn_id'].nunique())
print('Number of cardiac arrest encounters:', df['pat_enc_csn_id'].nunique())
print('Hospital admission is non elective:', df1['pat_enc_csn_id'].nunique())
print('Through ED or straight to ICU:', df2['pat_enc_csn_id'].nunique())
print('Diagnosis observed in ED or straight to ICU:', df3['pat_enc_csn_id'].nunique())
print(' duplicate osler_ids:', len(duplicate_osler_ids))
print('After removal of duplicates:', df4['pat_enc_csn_id'].nunique())

Cardiac arrest diagnoses: 8899
Number of cardiac arrest encounters: 4373
Hospital admission is non elective: 4036
Through ED or straight to ICU: 3751
Diagnosis observed in ED or straight to ICU: 2761
 duplicate osler_ids: 55
After removal of duplicates: 2649


In [24]:
ohca_df = df4.copy()
unique_osler_ids = pd.DataFrame(ohca_df['osler_id'])
unique_osler_ids.to_csv('ohca_osler_ids.csv',index=False)
unique_pat_enc_ids = pd.DataFrame(ohca_df['pat_enc_csn_id'])
unique_pat_enc_ids.to_csv('ohca_pat_enc_ids.csv',index=False)
ohca_df[['osler_id', 'pat_enc_csn_id', 'gender', 'birth_date', 'dx_name', 'hosp_admsn_time', 'hosp_disch_time', 'admit_source_c', 'disch_disp_c', 'ed_visit_yn']].to_csv('ohca.csv',index=False)

In [25]:
# icu_ohca_df = df4[df4['pat_enc_csn_id'].isin(icustay_df1['pat_enc_csn_id'])]
# unique_osler_ids = pd.DataFrame(icu_ohca_df['osler_id'])
# unique_osler_ids.to_csv('icu_ohca_osler_ids.csv',index=False)
# unique_pat_enc_ids = pd.DataFrame(icu_ohca_df['pat_enc_csn_id'])
# unique_pat_enc_ids.to_csv('icu_ohca_pat_enc_ids.csv',index=False)
# icu_ohca_df[['osler_id', 'pat_enc_csn_id', 'gender', 'birth_date', 'dx_name', 'hosp_admsn_time', 'hosp_disch_time', 'admit_source_c', 'disch_disp_c', 'ed_visit_yn']].to_csv('icu_ohca.csv',index=False)

In [26]:
ohca_df['pat_enc_csn_id'].nunique()

2649

In [27]:
ohca_df['osler_id'].nunique()

2649

In [28]:
ohca_df.shape

(2649, 27)